# NTFS Timestomping Detection Tool v2.0
# Step 1: Load Forensic Data

---

## Overview

This notebook loads forensic artifacts from NTFS filesystems:
- **$LogFile** events (NTFS transaction log)
- **$UsnJrnl** events (Update Sequence Number Journal)

These artifacts are merged to create a comprehensive timeline for analysis.

---

## Input Requirements

Provide paths to your raw forensic artifact CSV files in the configuration section below.

**$LogFile Required Columns**:
- `LSN` - LogFile sequence number
- `EventTime(UTC+8)` - Timestamp of event
- `File/Directory Name` - File name
- `Full Path` - Complete file path

**$UsnJrnl Required Columns**:
- `USN` - Update Sequence Number
- `TimeStamp(UTC+8)` - Timestamp of event  
- `File/Directory Name` - File name
- `FullPath` - Complete file path

The notebook will automatically normalize these column names for processing.

---

## 1. Setup

In [122]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


## 2. Configure Paths

In [123]:

# USER CONFIGURATION - EDIT THESE PATHS_______________________________________________________
# Input files - Full paths to your forensic artifact CSV files
LOGFILE_INPUT_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/raw/logfile/01-PE-LogFile.csv'
USNJRNL_INPUT_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/raw/usnjrnl/01-PE-UsnJrnl.csv'

# Output directory - Where to save processed data
OUTPUT_DIRECTORY = '/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/ntbk output'

# DO NOT EDIT BELOW THIS LINE__________________________________________________________________________________________

# Convert to Path objects
logfile_path = Path(LOGFILE_INPUT_PATH)
usnjrnl_path = Path(USNJRNL_INPUT_PATH)
OUTPUT_DIR = Path(OUTPUT_DIRECTORY)

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print("\nInput Files:")
print(f"  $LogFile:  {logfile_path}")
print(f"  $UsnJrnl:  {usnjrnl_path}")
print(f"\nOutput Directory:")
print(f"  {OUTPUT_DIR}")

print("\n" + "=" * 80)
print("FILE STATUS CHECK")
print("=" * 80)
print(f"\n$LogFile:  {'✓ Found' if logfile_path.exists() else '✗ Not found'}")
print(f"$UsnJrnl:  {'✓ Found' if usnjrnl_path.exists() else '✗ Not found'}")

if not logfile_path.exists() and not usnjrnl_path.exists():
    print("\nWARNING: No input files found!")
    print("    Please update the file paths in the configuration section above.")
elif not logfile_path.exists():
    print("\nNote: $LogFile not found - will process $UsnJrnl only")
elif not usnjrnl_path.exists():
    print("\nNote: $UsnJrnl not found - will process $LogFile only")
else:
    print("\n✓ Both files found - ready to proceed")

CONFIGURATION

Input Files:
  $LogFile:  /Users/soni/Github/Digital-Detectives_Thesis/data/raw/logfile/01-PE-LogFile.csv
  $UsnJrnl:  /Users/soni/Github/Digital-Detectives_Thesis/data/raw/usnjrnl/01-PE-UsnJrnl.csv

Output Directory:
  /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/ntbk output

FILE STATUS CHECK

$LogFile:  ✓ Found
$UsnJrnl:  ✓ Found

✓ Both files found - ready to proceed


## 3. Load Forensic Artifacts

In [124]:
print("=" * 80)
print("LOADING FORENSIC ARTIFACTS")
print("=" * 80)

# Load $LogFile events
print("\n1. Loading $LogFile events...")
if logfile_path.exists():
    logfile_df = pd.read_csv(logfile_path, encoding='utf-8-sig')
    print(f"   ✓ Loaded {len(logfile_df):,} $LogFile events")
    
    # Normalize column names for $LogFile
    column_mapping_lf = {
        'LSN': 'lf_lsn',
        'EventTime(UTC+8)': 'eventtime',
        'File/Directory Name': 'filename',
        'Full Path': 'filepath',
        'Event': 'lf_event',
        'Detail': 'lf_detail',
        'CreationTime': 'lf_creation_time',
        'ModifiedTime': 'lf_modified_time',
        'MFTModifiedTime': 'lf_mft_modified_time',
        'AccessedTime': 'lf_accessed_time',
        'Redo': 'lf_redo',
        'Target VCN': 'lf_target_vcn',
        'Cluster Index': 'lf_cluster_index'
    }
    
    # Only rename columns that exist in the dataframe
    rename_dict = {k: v for k, v in column_mapping_lf.items() if k in logfile_df.columns}
    logfile_df.rename(columns=rename_dict, inplace=True)
    print(f"   ✓ Normalized column names")
else:
    print("   ✗ $LogFile not found - skipping")
    logfile_df = pd.DataFrame()

# Load $UsnJrnl events  
print("\n2. Loading $UsnJrnl events...")
if usnjrnl_path.exists():
    usnjrnl_df = pd.read_csv(usnjrnl_path, encoding='utf-8-sig')
    print(f"   ✓ Loaded {len(usnjrnl_df):,} $UsnJrnl events")
    
    # Normalize column names for $UsnJrnl
    column_mapping_usn = {
        'USN': 'usn_usn',
        'TimeStamp(UTC+8)': 'eventtime',
        'File/Directory Name': 'filename',
        'FullPath': 'filepath',
        'EventInfo': 'usn_event_info',
        'SourceInfo': 'usn_source_info',
        'FileAttribute': 'usn_file_attribute',
        'Carving Flag': 'usn_carving_flag',
        'FileReferenceNumber': 'usn_file_ref_num',
        'ParentFileReferenceNumber': 'usn_parent_file_ref_num'
    }
    
    # Only rename columns that exist in the dataframe
    rename_dict = {k: v for k, v in column_mapping_usn.items() if k in usnjrnl_df.columns}
    usnjrnl_df.rename(columns=rename_dict, inplace=True)
    print(f"   ✓ Normalized column names")
else:
    print("   ✗ $UsnJrnl not found - skipping")
    usnjrnl_df = pd.DataFrame()

LOADING FORENSIC ARTIFACTS

1. Loading $LogFile events...
   ✓ Loaded 39,077 $LogFile events
   ✓ Normalized column names

2. Loading $UsnJrnl events...
   ✓ Loaded 316,817 $UsnJrnl events
   ✓ Normalized column names


## 4. Filter to Timestamp-Relevant Events

**Detection Patterns** (Oh et al., 2024):

**LogFile Events:**
- Time Reversal Event - Direct evidence of timestamp manipulation
- CreationTime Update Event - Explicit creation time modification

**UsnJrnl Events:**
- BASIC_INFO_CHANGE - Indicates $STANDARD_INFORMATION timestamp modification

**Important:** These filters match the exact patterns used during model training to ensure consistent detection behavior.

## 3B. Detect File System Tunneling (BEFORE Filtering)

**Important:** Tunneling detection must happen BEFORE filtering because it needs the full UsnJrnl dataset to look for delete/rename events within 15 seconds.

File System Tunneling occurs when:
1. A file is deleted/renamed/moved
2. Within 15 seconds, a NEW file with the same name is created
3. Windows caches and applies the old file's timestamps to the new file (benign behavior)

In [125]:
from datetime import timedelta

print("=" * 80)
print("FILE SYSTEM TUNNELING DETECTION")
print("=" * 80)

# We need to detect tunneling on the FULL UsnJrnl dataset before filtering
if not usnjrnl_df.empty:
    print(f"\nAnalyzing {len(usnjrnl_df):,} UsnJrnl events for tunneling patterns...")
    
    # Convert eventtime to datetime for time window calculations
    usnjrnl_df['eventtime'] = pd.to_datetime(usnjrnl_df['eventtime'], errors='coerce')
    
    # Create merge key for file matching
    usnjrnl_df['merge_key_temp'] = (usnjrnl_df['filepath'].fillna('').astype(str) + '|' +
                                     usnjrnl_df['filename'].fillna('').astype(str))
    
    # Initialize tunneling flag
    usnjrnl_df['is_tunneling'] = False
    
    # Find BASIC_INFO_CHANGE events (potential tunneling candidates)
    basic_info_events = usnjrnl_df[
        usnjrnl_df['usn_event_info'].fillna('').str.contains('Basic_Info_Change', case=False, na=False)
    ]
    
    print(f"  Found {len(basic_info_events):,} BASIC_INFO_CHANGE events to check")
    
    tunneling_count = 0
    
    # For each BASIC_INFO_CHANGE event, look for delete/rename within 15 seconds before
    for idx, row in basic_info_events.iterrows():
        if pd.isna(row['eventtime']):
            continue
            
        # Define 15-second window BEFORE this event
        time_window_start = row['eventtime'] - timedelta(seconds=15)
        time_window_end = row['eventtime']
        
        # Find prior events on same file within window
        prior_events = usnjrnl_df[
            (usnjrnl_df['merge_key_temp'] == row['merge_key_temp']) &
            (usnjrnl_df['eventtime'] >= time_window_start) &
            (usnjrnl_df['eventtime'] < time_window_end)
        ]
        
        # Check for tunneling indicators (delete/rename/move)
        tunneling_indicators = prior_events[
            prior_events['usn_event_info'].fillna('').str.contains(
                'File_Delete|Rename_Old_Name|Rename_New_Name',
                na=False,
                case=False,
                regex=True
            )
        ]
        
        if len(tunneling_indicators) > 0:
            usnjrnl_df.at[idx, 'is_tunneling'] = True
            tunneling_count += 1
    
    # Clean up temporary column
    usnjrnl_df.drop(columns=['merge_key_temp'], inplace=True)
    
    print(f"\n✓ Tunneling detection complete:")
    print(f"  Detected: {tunneling_count:,} events ({tunneling_count/len(usnjrnl_df)*100:.2f}%)")
    print(f"  Clean: {len(usnjrnl_df) - tunneling_count:,} events")
else:
    print("\nNo UsnJrnl data - skipping tunneling detection")

# Also initialize tunneling flag for LogFile (always False since tunneling is UsnJrnl-specific)
if not logfile_df.empty:
    logfile_df['is_tunneling'] = False

print("\n" + "=" * 80)

FILE SYSTEM TUNNELING DETECTION

Analyzing 316,817 UsnJrnl events for tunneling patterns...
  Found 24,002 BASIC_INFO_CHANGE events to check

✓ Tunneling detection complete:
  Detected: 445 events (0.14%)
  Clean: 316,372 events



## 4. Filter to Timestamp-Relevant Events

**Detection Patterns** (Oh et al., 2024):

**LogFile Events:**
- Time Reversal Event - Direct evidence of timestamp manipulation
- CreationTime Update Event - Explicit creation time modification

**UsnJrnl Events:**
- BASIC_INFO_CHANGE - Indicates $STANDARD_INFORMATION timestamp modification

**Important:** These filters match the exact patterns used during model training to ensure consistent detection behavior.

In [126]:
print("=" * 80)
print("DATA VALIDATION & FILTERING")
print("=" * 80)

required_columns = {
    'logfile': ['eventtime', 'filename', 'lf_lsn', 'lf_event'],
    'usnjrnl': ['eventtime', 'filename', 'usn_usn', 'usn_event_info']
}

# Validate & Filter $LogFile
if not logfile_df.empty:
    print("\n$LogFile validation:")
    missing_cols = [col for col in required_columns['logfile'] if col not in logfile_df.columns]
    if missing_cols:
        print(f"   ⚠️ Missing columns: {missing_cols}")
    else:
        print(f"   ✓ All required columns present")
        print(f"   ✓ Loaded: {len(logfile_df):,} total events")
        
        # Convert eventtime to datetime (if not already done by tunneling detection)
        if logfile_df['eventtime'].dtype != 'datetime64[ns]':
            logfile_df['eventtime'] = pd.to_datetime(logfile_df['eventtime'], errors='coerce')
        print(f"   ✓ Date range: {logfile_df['eventtime'].min()} to {logfile_df['eventtime'].max()}")
        
        # FILTER to timestamp manipulation events (matching training data)
        print(f"\n   Filtering to timestamp manipulation events...")
        
        # Time Reversal events (includes variations)
        time_reversal = logfile_df[
            logfile_df['lf_event'].str.contains('Time Reversal', na=False, case=False)
        ]
        
        # CreationTime Update Event
        creation_update = logfile_df[
            logfile_df['lf_event'].str.contains('CreationTime Update', na=False, case=False)
        ]
        
        logfile_df = pd.concat([time_reversal, creation_update]).drop_duplicates()
        
        print(f"   ✓ Time Reversal: {len(time_reversal):,} events")
        print(f"   ✓ CreationTime Update: {len(creation_update):,} events")
        print(f"   ✓ Filtered result: {len(logfile_df):,} events")

# Validate & Filter $UsnJrnl
if not usnjrnl_df.empty:
    print("\n$UsnJrnl validation:")
    missing_cols = [col for col in required_columns['usnjrnl'] if col not in usnjrnl_df.columns]
    if missing_cols:
        print(f"   ⚠️ Missing columns: {missing_cols}")
    else:
        print(f"   ✓ All required columns present")
        print(f"   ✓ Loaded: {len(usnjrnl_df):,} total events")
        
        # eventtime already converted to datetime by tunneling detection
        print(f"   ✓ Date range: {usnjrnl_df['eventtime'].min()} to {usnjrnl_df['eventtime'].max()}")
        
        # FILTER to BASIC_INFO_CHANGE events (preserve is_tunneling flag!)
        print(f"\n   Filtering to BASIC_INFO_CHANGE events...")
        usnjrnl_df = usnjrnl_df[
            usnjrnl_df['usn_event_info'].str.contains('Basic_Info_Change', na=False, case=False)
        ].copy()
        
        print(f"   ✓ Filtered result: {len(usnjrnl_df):,} events")
        print(f"   ✓ Preserved is_tunneling flag for {usnjrnl_df['is_tunneling'].sum()} events")

DATA VALIDATION & FILTERING

$LogFile validation:
   ✓ All required columns present
   ✓ Loaded: 39,077 total events
   ✓ Date range: 2022-12-16 17:14:27 to 2023-12-23 00:23:39

   Filtering to timestamp manipulation events...
   ✓ Time Reversal: 1,235 events
   ✓ CreationTime Update: 0 events
   ✓ Filtered result: 1,235 events

$UsnJrnl validation:
   ✓ All required columns present
   ✓ Loaded: 316,817 total events
   ✓ Date range: 2023-12-19 15:12:25 to 2023-12-23 00:23:49

   Filtering to BASIC_INFO_CHANGE events...
   ✓ Filtered result: 24,002 events
   ✓ Preserved is_tunneling flag for 445 events


## 4B. Create Merge Keys

Create merge keys for matching events across artifacts (filepath + filename).

In [127]:
print("=" * 80)
print("CREATING MERGE KEYS")
print("=" * 80)

# Create merge keys for matching (filepath + filename)
if not logfile_df.empty:
    logfile_df['merge_key'] = (logfile_df['filepath'].fillna('').astype(str) + '|' +
                                logfile_df['filename'].fillna('').astype(str))
    print(f"\n✓ Created {len(logfile_df):,} LogFile merge keys")

if not usnjrnl_df.empty:
    usnjrnl_df['merge_key'] = (usnjrnl_df['filepath'].fillna('').astype(str) + '|' +
                                usnjrnl_df['filename'].fillna('').astype(str))
    print(f"✓ Created {len(usnjrnl_df):,} UsnJrnl merge keys")

CREATING MERGE KEYS

✓ Created 1,235 LogFile merge keys
✓ Created 24,002 UsnJrnl merge keys


## 5. Smart Union Merging

**Smart Union Strategy** (Oh et al., 2024):
1. Match LogFile ↔ UsnJrnl events within ±1 second window for same file
2. Keep unmatched LogFile-only events (single-artifact evidence)
3. Keep unmatched UsnJrnl-only events (single-artifact evidence)
4. Combine all three types to preserve ALL forensic evidence

**Why Smart Union?**
- Prevents evidence loss (some timestomped events appear in only one artifact)
- Cross-artifact validation improves detection confidence
- Reduces data volume by merging duplicate events

In [128]:
print("=" * 80)
print("SMART UNION MERGING (±1 SECOND WINDOW)")
print("=" * 80)

from datetime import timedelta

# Step 1: Match LogFile <-> UsnJrnl within ±1 second window on same file
print("\n1. Matching LogFile <-> UsnJrnl (±1 second window)...")

matched_records = []
matched_lf_indices = set()
matched_usn_indices = set()

if not logfile_df.empty and not usnjrnl_df.empty:
    for lf_idx, lf_row in logfile_df.iterrows():
        # Find UsnJrnl events for same file within ±1 second
        potential_matches = usnjrnl_df[
            (usnjrnl_df['merge_key'] == lf_row['merge_key']) &
            (usnjrnl_df['eventtime'] >= lf_row['eventtime'] - timedelta(seconds=1)) &
            (usnjrnl_df['eventtime'] <= lf_row['eventtime'] + timedelta(seconds=1))
        ]
        
        if len(potential_matches) > 0:
            # Take closest match
            potential_matches = potential_matches.copy()
            potential_matches['time_diff'] = (potential_matches['eventtime'] - lf_row['eventtime']).abs()
            closest = potential_matches.nsmallest(1, 'time_diff').iloc[0]
            
            # Merge records
            merged_row = lf_row.copy()
            for col in usnjrnl_df.columns:
                if col not in merged_row.index and col not in ['eventtime', 'merge_key', 'filename', 'filepath']:
                    merged_row[col] = closest[col]
            
            merged_row['source'] = 'both'
            merged_row['has_logfile_evidence'] = True
            merged_row['has_usnjrnl_evidence'] = True
            # Preserve is_tunneling from UsnJrnl (LogFile is always False)
            merged_row['is_tunneling'] = closest.get('is_tunneling', False)
            matched_records.append(merged_row)
            
            matched_lf_indices.add(lf_idx)
            matched_usn_indices.add(closest.name)
    
    matched_df = pd.DataFrame(matched_records) if matched_records else pd.DataFrame()
    print(f"   Matched: {len(matched_df):,} records (source='both')")
else:
    matched_df = pd.DataFrame()
    print(f"   Matched: 0 records (missing one or both sources)")

# Step 2: Get unmatched LogFile records
print("\n2. Extracting unmatched LogFile records...")

if not logfile_df.empty:
    lf_only = logfile_df[~logfile_df.index.isin(matched_lf_indices)].copy()
    lf_only['source'] = 'logfile_only'
    lf_only['has_logfile_evidence'] = True
    lf_only['has_usnjrnl_evidence'] = False
    # Ensure is_tunneling exists (should be False from earlier)
    if 'is_tunneling' not in lf_only.columns:
        lf_only['is_tunneling'] = False
    print(f"   LogFile-only: {len(lf_only):,} records")
else:
    lf_only = pd.DataFrame()
    print(f"   LogFile-only: 0 records")

# Step 3: Get unmatched UsnJrnl records
print("\n3. Extracting unmatched UsnJrnl records...")

if not usnjrnl_df.empty:
    usn_only = usnjrnl_df[~usnjrnl_df.index.isin(matched_usn_indices)].copy()
    usn_only['source'] = 'usnjrnl_only'
    usn_only['has_logfile_evidence'] = False
    usn_only['has_usnjrnl_evidence'] = True
    # is_tunneling already exists from detection step
    print(f"   UsnJrnl-only: {len(usn_only):,} records")
else:
    usn_only = pd.DataFrame()
    print(f"   UsnJrnl-only: 0 records")

# Step 4: Combine all three types (Smart Union)
print("\n4. Creating smart union...")

if not matched_df.empty or not lf_only.empty or not usn_only.empty:
    combined_df = pd.concat([matched_df, lf_only, usn_only], ignore_index=True, sort=False)
    
    # Ensure is_tunneling column exists in final output
    if 'is_tunneling' not in combined_df.columns:
        combined_df['is_tunneling'] = False
    
    print(f"\n✓ Smart union complete:")
    print(f"   Total events: {len(combined_df):,}")
    print(f"   LogFile only: {(combined_df['source'] == 'logfile_only').sum():,}")
    print(f"   UsnJrnl only: {(combined_df['source'] == 'usnjrnl_only').sum():,}")
    print(f"   Both sources: {(combined_df['source'] == 'both').sum():,}")
    
    # Show tunneling stats
    tunneling_in_output = combined_df['is_tunneling'].sum()
    print(f"   Tunneling events: {tunneling_in_output:,} ({tunneling_in_output/len(combined_df)*100:.2f}%)")
    
    # Data reduction summary
    input_total = len(logfile_df) + len(usnjrnl_df)
    output_total = len(combined_df)
    reduction = (input_total - output_total) / input_total * 100 if input_total > 0 else 0
    print(f"\n   Input: {len(logfile_df):,} LogFile + {len(usnjrnl_df):,} UsnJrnl = {input_total:,} total")
    print(f"   Output: {output_total:,} merged events")
    print(f"   Reduction: {reduction:.1f}% (eliminated {input_total - output_total:,} duplicates)")
else:
    print("\n✗ ERROR: No data to merge!")
    combined_df = pd.DataFrame()

SMART UNION MERGING (±1 SECOND WINDOW)

1. Matching LogFile <-> UsnJrnl (±1 second window)...
   Matched: 1,053 records (source='both')

2. Extracting unmatched LogFile records...
   LogFile-only: 182 records

3. Extracting unmatched UsnJrnl records...
   UsnJrnl-only: 22,969 records

4. Creating smart union...

✓ Smart union complete:
   Total events: 24,204
   LogFile only: 182
   UsnJrnl only: 22,969
   Both sources: 1,053
   Tunneling events: 445 (1.84%)

   Input: 1,235 LogFile + 24,002 UsnJrnl = 25,237 total
   Output: 24,204 merged events
   Reduction: 4.1% (eliminated 1,033 duplicates)


## 6. Save Merged Data

In [129]:
print("=" * 80)
print("SAVING MERGED DATA")
print("=" * 80)

if not combined_df.empty:
    output_file = OUTPUT_DIR / 'merged_forensic_data.csv'
    combined_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n✓ Saved merged data to:")
    print(f"   {output_file}")
    print(f"\n   Total records: {len(combined_df):,}")
    print(f"   Total columns: {len(combined_df.columns)}")
    print(f"\n✓ Ready for feature engineering (Step 2)")
else:
    print("\n✗ No data to save - check input files")

SAVING MERGED DATA

✓ Saved merged data to:
   /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/01-PE/ntbk output/merged_forensic_data.csv

   Total records: 24,204
   Total columns: 25

✓ Ready for feature engineering (Step 2)


## 7. Data Summary

In [130]:
if not combined_df.empty:
    print("=" * 80)
    print("DATA SUMMARY")
    print("=" * 80)
    
    print(f"\nFirst 5 records:")
    display(combined_df.head())
    
    print(f"\nColumn types:")
    print(combined_df.dtypes)
    
    print(f"\nMissing data:")
    missing = combined_df.isnull().sum()
    print(missing[missing > 0])

DATA SUMMARY

First 5 records:


,lf_lsn,eventtime,lf_event,lf_detail,filename,filepath,lf_creation_time,lf_modified_time,lf_mft_modified_time,lf_accessed_time,lf_redo,lf_target_vcn,lf_cluster_index,is_tunneling,merge_key,usn_usn,usn_event_info,usn_source_info,usn_file_attribute,usn_carving_flag,usn_file_ref_num,usn_parent_file_ref_num,source,has_logfile_evidence,has_usnjrnl_evidence
0,8.713792e+09,2023-12-23 00:14:23,Time Reversal Event,ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-...,dropboxstatus-connecting@3x.png,\Program Files (x86)\Dropbox\Client\189.4.8395...,NaN,NaN,NaN,NaN,Update Resident Value,0x170EF,6.0,False,\Program Files (x86)\Dropbox\Client\189.4.8395...,1.325016e+09,Basic_Info_Changed,Normal,Archive,NaN,0x000800000005C3BF,0x000600000005C36C,both,True,True
1,8.713793e+09,2023-12-23 00:14:23,Time Reversal Event,ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-...,dropboxstatus-idle.png,\Program Files (x86)\Dropbox\Client\189.4.8395...,NaN,NaN,NaN,NaN,Update Resident Value,0x170F0,0.0,False,\Program Files (x86)\Dropbox\Client\189.4.8395...,1.325017e+09,Basic_Info_Changed,Normal,Archive,NaN,0x000800000005C3C0,0x000600000005C36C,both,True,True
2,8.713794e+09,2023-12-23 00:14:23,Time Reversal Event,ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-...,dropboxstatus-idle@1p25x.png,\Program Files (x86)\Dropbox\Client\189.4.8395...,NaN,NaN,NaN,NaN,Update Resident Value,0x171DC,2.0,False,\Program Files (x86)\Dropbox\Client\189.4.8395...,1.325017e+09,Basic_Info_Changed,Normal,Archive,NaN,0x001C00000005C771,0x000600000005C36C,both,True,True
3,8.713794e+09,2023-12-23 00:14:23,Time Reversal Event,ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-...,dropboxstatus-idle@1p5x.png,\Program Files (x86)\Dropbox\Client\189.4.8395...,NaN,NaN,NaN,NaN,Update Resident Value,0x1727B,0.0,False,\Program Files (x86)\Dropbox\Client\189.4.8395...,1.325018e+09,Basic_Info_Changed,Normal,Archive,NaN,0x001100000005C9EC,0x000600000005C36C,both,True,True
4,8.713795e+09,2023-12-23 00:14:23,Time Reversal Event,ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-...,dropboxstatus-idle@1p75x.png,\Program Files (x86)\Dropbox\Client\189.4.8395...,NaN,NaN,NaN,NaN,Update Resident Value,0x17281,4.0,False,\Program Files (x86)\Dropbox\Client\189.4.8395...,1.325019e+09,Basic_Info_Changed,Normal,Archive,NaN,0x000900000005CA06,0x000600000005C36C,both,True,True



Column types:
lf_lsn                            float64
eventtime                  datetime64[ns]
lf_event                           object
lf_detail                          object
filename                           object
filepath                           object
lf_creation_time                   object
lf_modified_time                   object
lf_mft_modified_time               object
lf_accessed_time                   object
lf_redo                            object
lf_target_vcn                      object
lf_cluster_index                  float64
is_tunneling                         bool
merge_key                          object
usn_usn                           float64
usn_event_info                     object
usn_source_info                    object
usn_file_attribute                 object
usn_carving_flag                  float64
usn_file_ref_num                   object
usn_parent_file_ref_num            object
source                             object
has_logfile_evidenc

---

## ✓ Step 1 Complete!

**Next:** Run `02_Feature_Engineering.ipynb` to extract forensic features from the merged data.

---